#Notebook da Camada Silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

#tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]

df_bronze_chamados = spark.read.table(f"{bronze_schema}.chamados")

In [0]:
def normalizar_texto(col):
    r"""
    Normaliza texto:
    - tudo minúsculo
    - mantém só letras (com acento) e números
    - todo o resto (pontuação, símbolos, '_', \n, \t, \r e outros espaços) vira espaço
    - sequências de espaço viram um só, e trim nas pontas
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    return F.trim(
        F.regexp_replace(F.lower(c.cast("string")), r"[^\p{L}\p{M}0-9]+", " ")
    )

In [0]:
def normalizar_inteiro(col):
    """
    Remove tudo que não for dígito (0-9) e converte para inteiro (bigint).
    - texto sem nenhum dígito (ex.: 'NULL', '', 'abc') vira null
    - mais de 18 dígitos vira null (evita estouro do bigint)
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    digitos = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return (
        F.when((digitos == "") | (F.length(digitos) > 18), F.lit(None).cast("long"))
         .otherwise(digitos.cast("long"))
    )

In [0]:
# dd/mm/aaaa  <qualquer coisa que não seja dígito>  hh:mm
# o \D+ cobre " às ", " �s ", " " etc. — não depende do encoding do "à"
PADRAO_DATAHORA = r"(\d{1,2})/(\d{1,2})/(\d{4})\D+(\d{1,2}):(\d{2})"

def extrair_datahora(col, prefixo):
    """
    Extrai dia, mês, ano, hora e minuto de strings tipo '12/07/2025 às 09:23:50'.
    Retorna um dict {f'{prefixo}_dia': Column, ...} para usar com df.withColumns().
    Se a string não bater com o padrão, as 5 partes vêm null.
    """
    c = F.col(col) if isinstance(col, str) else col
    s = c.cast("string")
    partes = ["dia", "mes", "ano", "hora", "minuto"]
    return {
        f"{prefixo}_{nome}": normalizar_inteiro(F.regexp_extract(s, PADRAO_DATAHORA, i))
        for i, nome in enumerate(partes, start=1)
    }

In [0]:
# ======================= auxiliares =======================
TOKENS_NULOS = ["NULL", "NONE", "NAN", "N/A"]
ACENTOS      = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM_ACENTOS  = "aaaaaeeeeiiiiooooouuuuc"

def mostrar(df_, descricao):
    """Só imprime a descrição e faz o display se o DataFrame tiver linhas."""
    if not df_.isEmpty():
        print(f"\n{descricao}")
        display(df_)

def chave_comparacao(col):
    """normalizar_texto + sem acento. Usada só para COMPARAR categorias, não vai para a tabela."""
    return F.translate(F.regexp_replace(normalizar_texto(col), r"\p{M}", ""), ACENTOS, SEM_ACENTOS)

def somente_digitos(col):
    """Mantém só 0-9, como string (preserva zero à esquerda). Sem nenhum dígito -> null."""
    c = F.col(col) if isinstance(col, str) else col
    d = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return F.when(d == "", F.lit(None).cast("string")).otherwise(d)

def normalizar_tempo(col):
    """Segundos como bigint: descarta tudo a partir do primeiro '.' ou ',' e remove
    qualquer não-dígito (inclusive o sinal). Sem dígitos -> null."""
    c = F.col(col) if isinstance(col, str) else col
    parte_inteira = F.regexp_extract(c.cast("string"), r"^[^.,]*", 0)
    return normalizar_inteiro(parte_inteira)

def padronizar_categoria(df, coluna, mapa, limiar=0.8):
    """
    mapa = {valor_canonico: [variantes aceitas]}   (canônico None => vira null)
    Para cada valor distinto da coluna, acha a variante mais parecida:
      similaridade = 1 - levenshtein / maior comprimento   (1.0 = idêntico)
    similaridade >= limiar -> valor canônico ; abaixo disso, vazio ou null -> null
    Retorna (df com a coluna padronizada, tabela de mapeamento para auditoria).
    """
    pares = [(can, var) for can, variantes in mapa.items()
                        for var in ([can] if can is not None else []) + variantes]
    ref = (spark.createDataFrame(pares, "canonico string, variante string")
                .select(chave_comparacao("variante").alias("chave_ref"), "canonico")
                .distinct())

    mapeamento = (
        df.select(F.col(coluna).alias("valor_original"), chave_comparacao(coluna).alias("chave"))
          .filter(F.col("chave").isNotNull() & (F.col("chave") != ""))
          .distinct()
          .crossJoin(ref)
          .withColumn("similaridade",
                      1 - F.levenshtein("chave", "chave_ref") / F.greatest(F.length("chave"), F.length("chave_ref")))
          .groupBy("valor_original")
          .agg(F.max_by("canonico", "similaridade").alias("mais_parecido"),
               F.round(F.max("similaridade"), 3).alias("similaridade"))
          .withColumn("valor_padronizado",
                      F.when(F.col("similaridade") >= limiar, F.col("mais_parecido")))
    )

    df_out = (
        df.join(F.broadcast(mapeamento.select(F.col("valor_original").alias(coluna), "valor_padronizado")),
                on=coluna, how="left")
          .withColumn(coluna, F.col("valor_padronizado"))
          .select(*df.columns)
    )
    return df_out, mapeamento

# ======================= listas de referência =======================
# variações só de maiúscula/acento já caem na mesma chave; ficam listadas como documentação
MOTIVOS = {
    "Consulta de fatura":               ["CONSULTA DE FATURA"],
    "Problema com aplicativo":          ["PROBLEMA COM APLICATIVO"],
    "Consulta de limite":               ["CONSULTA DE LIMITE"],
    "Bloqueio de cartão":               ["BLOQUEIO DE CARTÃO", "Bloqueio de cartao"],
    "Compra não autorizada":            ["COMPRA NÃO AUTORIZADA", "Compra nao autorizada"],
    "Renegociação de dívida":           ["RENEGOCIAÇÃO DE DÍVIDA", "Renegociacao de divida"],
    "Contestação de fatura":            ["CONTESTAÇÃO DE FATURA", "Contestacao de fatura"],
    "Desbloqueio de cartão":            ["DESBLOQUEIO DE CARTÃO", "Desbloqueio de cartao"],
    "Cancelamento de conta":            ["CANCELAMENTO DE CONTA"],
    "Consulta de contrato":             ["CONSULTA DE CONTRATO"],
    "Dúvidas gerais sobre programa de pontos": ["DÚVIDAS GERAIS SOBRE PROGRAMA DE PONTOS",
                                                "Duvidas gerais sobre programa de pontos"],
    "Contratação de cartão adicional":  ["CONTRATAÇÃO DE CARTÃO ADICIONAL", "Contratacao de cartao adicional"],
    "Alteração de dados cadastrais (vencimento da fatura, telefone, email)": [
        "ALTERAÇÃO DE DADOS CADASTRAIS (VENCIMENTO DA FATURA, TELEFONE, EMAIL)",
        "Alteracao de dados cadastrais (vencimento da fatura, telefone, email)",
    ],
    "Transferência de agência":         ["TRANSFERÊNCIA DE AGÊNCIA"],
}

CANAIS = {
    "URA":                       ["ura", "Ura", "U.R.A"],
    "Chatbot":                   ["ChatBot", "chat bot", "chatbot", "CHATBOT"],
    "Atendimento Inicial":       ["Atendimento_Inicial", "atendimento inicial", "ATENDIMENTO INICIAL", "Atend. Inicial"],
    "Atendimento Especializado": ["Atendimento_Especializado", "Atend. Especializado",
                                  "atendimento especializado", "ATENDIMENTO ESPECIALIZADO"],
    None:                        ["null"],
}

RESOLVIDO = {
    "Sim": ["S", "1"],
    "Não": ["N", "0", "Nao resolvido"],
    None:  ["null"],
}

# ======================= 0. diagnóstico: datas nas colunas categóricas =======================
print("=== DIAGNÓSTICO: datas em motivo / canal / resolvido (bronze, antes da limpeza) ===")
PADRAO_DATA = r"\d{1,2}/\d{1,2}/\d{2,4}"
tem_data = {c: F.col(c).cast("string").rlike(PADRAO_DATA) for c in ["motivo", "canal", "resolvido"]}
for c, cond in tem_data.items():
    print(f"  {c}: {df_bronze_chamados.filter(cond).count()} linhas com data")
mostrar(
    df_bronze_chamados.filter(tem_data["motivo"] | tem_data["canal"] | tem_data["resolvido"]),
    "Linhas da bronze com data em motivo, canal ou resolvido. Confira se os vizinhos estão deslocados "
    "(ex.: valor de canal em motivo, de resolvido em canal, tempos fora do lugar)",
)

# ======================= 1. remove colunas de hora =======================
df = df_bronze_chamados.drop("hora_inicio_atendimento", "hora_abertura_chamado", "hora_finalizacao_atendimento")

# ======================= 2. padroniza categorias =======================
df, map_motivo    = padronizar_categoria(df, "motivo", MOTIVOS)
df, map_canal     = padronizar_categoria(df, "canal", CANAIS)
df, map_resolvido = padronizar_categoria(df, "resolvido", RESOLVIDO)

df = df.withColumn("resolvido",                      # Sim/Não -> boolean (o resto fica null)
                   F.when(F.col("resolvido") == "Sim", F.lit(True))
                    .when(F.col("resolvido") == "Não", F.lit(False)))

# ======================= 3. ids (string só dígitos) e tempos (bigint) =======================
for c in ["id_chamado", "id_cliente", "id_atendente"]:
    df = df.withColumn(c, somente_digitos(c))
for c in ["tempo_espera", "tempo_atendimento"]:
    df = df.withColumn(c, normalizar_tempo(c))

# ======================= 4. 'null' escrito -> null de verdade =======================
df = df.withColumns({
    c: F.when((F.trim(F.col(c)) == "") | F.upper(F.trim(F.col(c))).isin(TOKENS_NULOS),
              F.lit(None).cast("string")).otherwise(F.col(c))
    for c, tipo in df.dtypes if tipo == "string"
})

# ======================= 5. duplicidade por id_chamado =======================
print("\n=== DUPLICIDADE POR id_chamado ===")
w_id = Window.partitionBy("id_chamado")

# 5a. remove linha repetida com motivo ou canal nulo, se o mesmo id tiver outra linha completa
df = (
    df.withColumn("_ocorrencias", F.count("*").over(w_id))
      .withColumn("_incompleta", F.col("motivo").isNull() | F.col("canal").isNull())
      .withColumn("_completas_no_id", F.sum(F.when(~F.col("_incompleta"), 1).otherwise(0)).over(w_id))
)
remover = (F.col("id_chamado").isNotNull() & (F.col("_ocorrencias") > 1)
           & F.col("_incompleta") & (F.col("_completas_no_id") > 0))
print(f"Linhas removidas (id repetido com motivo ou canal nulo): {df.filter(remover).count()}")
df = df.filter(~remover).drop("_ocorrencias", "_incompleta", "_completas_no_id")

# 5b. remove cópias exatas (ignorando o metadado de ingestão)
colunas_negocio = [c for c in df.columns if c != "ingestion_datetime"]
antes = df.count()
df = df.dropDuplicates(colunas_negocio)
print(f"Cópias exatas removidas: {antes - df.count()}")

# 5c. o que sobrou repetido diverge em conteúdo -> só exibe
duplicatas_restantes = (vcredit_medallion.silver.tb_chamados
    df.withColumn("_ocorrencias", F.count("*").over(w_id))
      .filter(F.col("id_chamado").isNotNull() & (F.col("_ocorrencias") > 1))
      .drop("_ocorrencias")
      .orderBy("id_chamado")
)
mostrar(duplicatas_restantes, "id_chamado ainda repetidos, com conteúdo divergente (todas as linhas de cada id)")

# ======================= 6. motivo/canal ainda nulos =======================
print("\n=== motivo OU canal NULOS APÓS A LIMPEZA ===")
df_motivo_canal_nulos = df.filter(F.col("motivo").isNull() | F.col("canal").isNull())
mostrar(df_motivo_canal_nulos, f"Linhas com motivo ou canal nulo ({df_motivo_canal_nulos.count()} linhas)")

df_silver_chamados = df

# ======================= 7. grava na silver =======================
(df_silver_chamados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados"))
print(f"\nGravado em {silver_schema}.tb_chamados")